# Engine: The Box-Kite Debugger — the ZD geometry, made watchable

**File:** `ValaQuenta/modules/box_kite/`
**ValaQuenta wiki:** [wiki/box_kite.md](../../wiki/box_kite.md)
**Ainulindale wiki:** `Ainulindale/wiki/84_the_box_kite_debugger.md`

> *"how do we 'debug' the geometries / how do we watch the geometries interact"*
> — Cody Michael Allison, 2026-08-05

## Where the object is, and where it is not

Moreno (1997) proved the sedenions' norm-one zero divisors are homeomorphic to
the exceptional Lie group **G₂**. That is true, and it is the wrong place to
build. de Marrais (2000), responding directly:

> *"Moreno discovered a homomorphism — a 'blow-up' of an exact correspondence —
> and the 'blow-ups' in the history of number theory have all entailed the loss
> of something."*

G₂ is the **continuous shadow**. It forgets which Fano line is which. The exact
object is finite:

    PSL(2,7),  order 168,  = Aut(Fano plane) = GL(3,2)

PSL(2,7) is the finite subgroup of G₂ that **preserves the labelling**. Every
structure below is exactly enumerable — no sampling, no fitting. That exactness
is the entire point of a debugger.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math, itertools
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

from ValaQuenta.modules.box_kite import (
    basis_mul, multiply, is_zero, basis_vector, associator, commutator,
    associator_defect, associator_census, diagonals, is_assessor, assessors,
    strut, box_kites, zero_divisor_pairs, verify_counts, assessors_adjacent,
    box_kite_graph, chart_spectrum, glued_graph, glued_spectrum,
    associator_field, pg32_points, pg32_lines, fano_planes, psl27_order,
    skeleton_counts, e0_is_outside,
)
print('python', sys.version.split()[0])

---
## 1. The honest check

Every count below is **derived from the Cayley–Dickson multiplication table** in
`maths.py`. Nothing is read in from de Marrais. Agreement with his published
values — and with ValaQuenta's own `ZD_PAIRS=84`, `ZD_CLASSES=42`,
`ZD_COMPOSITE=168` — is a **check**, not an input.

A mismatch would be a bug in this module, not a discovery.

In [ ]:
v = verify_counts()
for k, val in v.items():
    print(f"  {k:<26} {val}")
print()
print("  skeleton (PG(3,2)):")
for k, val in skeleton_counts().items():
    print(f"    {k:<22} {val}")

### How the counts arise

    ASSESSOR    a plane span(e_a, e_{b+8}) with a,b ∈ 1..7 whose diagonals
                e_a ± e_{b+8} zero-divide.  a == b NEVER works.
                49 − 7 = 42 Assessors
    84          42 Assessors × 2 diagonals
    168         42 × 4 signed unit points = |PSL(2,7)|
    336         ordered annihilating pairs = 84 × 4
                (each diagonal annihilates exactly 4 others)
    STRUT       s = a XOR b ∈ 1..7 — indexes the box-kite
    7 × 6 = 42  seven box-kites, six Assessors each

In [ ]:
for s, members in sorted(box_kites().items()):
    print(f"  strut {s}:  {members}")
print()
print("  de Marrais Box-Kite I is (3,10),(2,11),(5,12),(4,13),(7,14),(6,15)")
print("  → (a,b) = (3,2),(2,3),(5,4),(4,5),(7,6),(6,7)")
print(f"  → this module's strut 1: {sorted(box_kites()[1])}")
print(f"  → match: {sorted(box_kites()[1]) == sorted([(3,2),(2,3),(5,4),(4,5),(7,6),(6,7)])}")

---
## 2. THE SHAPE IS AN OCTAHEDRON

For each strut the 6 Assessors form a 4-regular graph on 6 vertices with exactly
**3 non-edges** — and the non-edges are precisely the reversal pairs
(a,b) ↔ (b,a). That is **K₂,₂,₂, the octahedron**.

Built from actual vanishing products, not imposed.

In [ ]:
for s in range(1, 8):
    g = box_kite_graph(s)
    print(f"  strut {s}: {len(g['edges'])} edges, degrees {g['degrees']}, "
          f"octahedron={g['is_octahedron']}, non-edges are reversals={g['non_edges_are_reversals']}")
print()
g = box_kite_graph(1)
print("  strut 1 vertices :", g['vertices'])
print("  strut 1 non-edges:", [(g['vertices'][i], g['vertices'][j]) for i, j in g['non_edges']])

In [ ]:
# The seven charts, drawn as octahedra
OCT = np.array([[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1]], float)
FACES = [(0,2,4),(2,1,4),(1,3,4),(3,0,4),(0,2,5),(2,1,5),(1,3,5),(3,0,5)]

fig = plt.figure(figsize=(14, 7))
for n, s in enumerate(range(1, 8)):
    g = box_kite_graph(s)
    V, fld = g['vertices'], associator_field(s)['vertex_defect']
    # antipodal pairs (the reversals) placed on opposite octahedron vertices
    order, used = [], set()
    for i, j in g['non_edges']:
        order += [i, j]; used |= {i, j}
    order += [i for i in range(6) if i not in used]
    pos = {order[k]: OCT[k] for k in range(6)}
    ax = fig.add_subplot(2, 4, n+1, projection='3d')
    ax.add_collection3d(Poly3DCollection([[OCT[a],OCT[b],OCT[c]] for a,b,c in FACES],
                        alpha=0.10, facecolor='#47c', edgecolor='0.6', linewidths=0.6))
    d = np.array([fld[V[i]] for i in range(6)])
    P = np.array([pos[i] for i in range(6)])
    ax.scatter(P[:,0], P[:,1], P[:,2], c=d, cmap='inferno', s=90,
               edgecolor='k', linewidth=0.4, depthshade=False)
    for i in range(6):
        ax.text(*(P[i]*1.35), f"{V[i][0]},{V[i][1]+8}", fontsize=6, ha='center')
    ax.set_title(f"strut {s}", fontsize=9); ax.set_axis_off()
    ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
fig.suptitle('The seven box-kites — colour = associator defect (the curvature)', y=0.98)
plt.tight_layout(); plt.show()

---
## 3. The dispersion relation, chart level

The octahedral graph Laplacian spectrum is closed form:

    adjacency:   4,  0,  0,  0, −2, −2
    Laplacian:   0,  4,  4,  4,  6,  6      ← ω²(k) on one box-kite

One zero mode, a 3-fold degenerate mode at 4, a 2-fold at 6.

**The zero mode is e₀'s signature** — the mode that exists everywhere and
propagates nowhere. It emerges from the graph; it is not inserted.

In [ ]:
for s in range(1, 8):
    print(f"  strut {s}: {[round(x, 9) for x in chart_spectrum(s)]}")
print()
print("  every chart carries exactly one zero mode — e_0's signature")

### 0_RB is not the geometry — checked, not asserted

e₀ is not a point of PG(3,2), is in no Assessor, is a vertex of no box-kite, and
its associator vanishes against everything. It **generates the boundary and does
not live on it.**

In [ ]:
for k, val in e0_is_outside().items():
    print(f"  {k:<34} {val}")
print()
print("  census of curvature:", associator_census())

---
## 4. The atlas — and the result that changes the open problem

Assembling all 42 Assessors into one graph produces something worth stopping on:
**there are no cross-strut edges at all.** The seven charts are mutually
disconnected under zero-divisor adjacency.

That is not a failure of the instrument — it is a finding, and it sharpens the
open problem. A wave cannot propagate between charts via ZD adjacency, so either

1. the medium genuinely is seven disconnected octahedra and there is no global
   dispersion relation to find, or
2. the connection between charts is the **PSL(2,7) group action permuting the
   struts**, not an adjacency — i.e. the transition maps are group elements,
   not edges.

(2) is where I would look. Either way, the gluing question has changed shape:
it is no longer "find the edges between charts" but "find the group action that
identifies them."

In [ ]:
gg = glued_graph()
print(f"  vertices          {gg['n_vertices']}")
print(f"  edges             {gg['n_edges']}   (= 7 charts × 12)")
print(f"  within-strut      {gg['within_strut_edges']}")
print(f"  CROSS-STRUT       {gg['cross_strut_edges']}      <-- the finding")
print(f"  degrees uniform   {len(set(gg['degrees'])) == 1} (all {gg['degrees'][0]})")
print()
spec = glued_spectrum()
print("  glued spectrum:", [round(x, 6) for x in spec])
print(f"  zero modes: {sum(1 for x in spec if abs(x) < 1e-9)}  "
      f"(one per disconnected chart — the graph-theoretic signature of disconnection)")
print()
print("  NOTE: 84 = ZD_PAIRS is ALSO the edge count of the atlas.")
print("        42 vertices at degree 4 → 42×4/2 = 84. Same number, second reading.")

---
## 5. The skeleton: PG(3,2)

The 15 pure imaginaries are the 15 points of the finite projective tetrahedron,
with 35 lines of 3 (each a multiplication triplet) and **15** Fano planes.

Not 32 — figures circulating with "32 interlocking Fano planes" are wrong.

In [ ]:
print("  points:", pg32_points())
print(f"  lines: {len(pg32_lines())}  first ten: {pg32_lines()[:10]}")
print(f"  Fano planes: {len(fano_planes())}, each of size {len(fano_planes()[0])}")
print(f"  |PSL(2,7)| = {psl27_order()} = the primitive unit ZD count")
print()
fig, ax = plt.subplots(figsize=(7, 7))
th = np.linspace(0, 2*np.pi, 16)[:15] + np.pi/2
xy = {p: (np.cos(th[p-1]), np.sin(th[p-1])) for p in range(1, 16)}
for a, b, c in pg32_lines():
    for u, v in ((a,b),(b,c),(a,c)):
        ax.plot([xy[u][0], xy[v][0]], [xy[u][1], xy[v][1]], color='#47c', lw=0.35, alpha=0.5)
for p, (x, y) in xy.items():
    ax.plot(x, y, 'o', color='0.15', ms=9)
    ax.text(x*1.13, y*1.13, str(p), ha='center', va='center', fontsize=9)
ax.set_aspect('equal'); ax.set_axis_off()
ax.set_title('PG(3,2): 15 points, 35 lines — the skeleton of the ZD geometry')
plt.tight_layout(); plt.show()

---
## What is open

**The gluing.** Each chart is exactly computable; the curvature of the atlas
lives in the transitions between the 7 box-kites, and section 4 shows those
transitions are *not* edges — there are none. The transition maps have to come
from the PSL(2,7) action on the struts, and they are not yet written.

Until they are, `glued_graph()` / `glued_spectrum()` are an **instrument
reading**, not a derivation of the global dispersion relation. That distinction
is stated in the module docstring and repeated here so it cannot be read past.